### Silver to Gold Notebook 

__Import library__

In [2]:
import logging
from pyspark.sql.utils import AnalysisException
from pyspark.sql import functions as F
import pandas as pd

logging.basicConfig(
    level=logging.INFO, 
    format="%(asctime)s - %(levelname)s - %(message)s") ## python logger
logger = logging.getLogger("silver_to_gold_pipeline.log") ##pyspark logger

StatementMeta(, 305dd33b-06f1-4eb7-854a-9828f7493c0a, 4, Finished, Available, Finished, False)

__Import function__

In [3]:
logging.disable(logging.CRITICAL)

StatementMeta(, 305dd33b-06f1-4eb7-854a-9828f7493c0a, 5, Finished, Available, Finished, False)

In [4]:
%run Silver_to_Gold Notebook_function_unit_test

StatementMeta(, 305dd33b-06f1-4eb7-854a-9828f7493c0a, 12, Finished, Available, Finished, True)

In [5]:
logging.disable(logging.NOTSET)

StatementMeta(, 305dd33b-06f1-4eb7-854a-9828f7493c0a, 13, Finished, Available, Finished, False)

---
__Create dim_season__  
- using original season column as season key because it is unlikely the current season key will change value.
- Table is meant to be rerun or replace again when fed with new data.

In [6]:
logger.info("Creating gold_dim_season table...")

gold_dim_season_table_query = """
    CREATE OR REPLACE TABLE nhl_gold_lakehouse.dbo.gold_dim_season AS
    SELECT
        DISTINCT CAST(season AS INT) AS season_key, --- using original 'season' as season_key
        CAST(LEFT(season,4) || '-' || RIGHT(season, 4) AS STRING) AS season_label,
        CAST(LEFT(season, 4) AS INT) AS start_year,
        CAST(RIGHT(season, 4) AS INT) AS end_year
    FROM nhl_silver_lakehouse.dbo.silver_game"""

try:
    ## Table creation
    spark.sql(gold_dim_season_table_query)
    logger.info("Query executed")
except AnalysisException as e:
    logger.error(f"ANALYSIS EXCEPTION ERROR: Check source table or column name. \n\n Details:{e} ") ## this one catch pyspark error
    raise ## reraise it
except Exception as e:
    logger.error(f"UNEXPECTED ERROR:\n\n {e}.", exc_info=True) ## this one catch python other error
    raise ## reraise it


StatementMeta(, 305dd33b-06f1-4eb7-854a-9828f7493c0a, 14, Finished, Available, Finished, False)

2026-10-04 14:44:35,449 - INFO - Creating gold_dim_season table...
2026-10-04 14:45:02,334 - INFO - Query executed


__Validate dim_season__

In [7]:
pri_key_validation("nhl_gold_lakehouse.dbo.gold_dim_season", "season_key")

StatementMeta(, 305dd33b-06f1-4eb7-854a-9828f7493c0a, 15, Finished, Available, Finished, False)

2026-10-04 14:45:23,890 - INFO - Prepare to validate nhl_gold_lakehouse.dbo.gold_dim_season...
2026-10-04 14:45:24,045 - INFO - 1/3 VALIDATION SUCCESS: nhl_gold_lakehouse.dbo.gold_dim_season is created successfully.
2026-10-04 14:45:24,046 - INFO - 1/3 - Existence Check completed
2026-10-04 14:45:28,246 - INFO - 2/3 VALIDATION SUCCESS: season_key is unique
2026-10-04 14:45:28,251 - INFO - 2/3 - Unique check completed.
2026-10-04 14:45:30,146 - INFO - 3/3 VALIDATION SUCCESS: No nulls in season_key
2026-10-04 14:45:30,148 - INFO - 3/3 - Null check completed.
2026-10-04 14:45:30,149 - INFO - Full validation completed.


---
__Create dim_team__  
- Dropping off venue link as it is irrelevant to venue
- Table is meant to be rerun or replace again when fed with new data.

In [8]:
logger.info("Creating gold_dim_table...")

gold_dim_team_query = """
CREATE OR REPLACE TABLE nhl_gold_lakehouse.dbo.gold_dim_team AS
SELECT
    CAST(team_id AS INT),
    CAST(franchise_id AS INT),
    CAST(short_name AS STRING),
    CAST(team_name AS STRING),
    CAST(abbreviation AS STRING)
FROM nhl_silver_lakehouse.dbo.silver_team_info
"""

try:
    ## Table creation
    spark.sql(gold_dim_team_query)
    logger.info("Query executed")

except AnalysisException as e:
    logger.error(f"ANALYSIS EXCEPTION ERROR: Check source table or column name. \n\n Details:{e}")
    raise
except Exception as e:
    logger.info(f"UNEXPECTED ERROR:\n\n {e}.", exc_info=True)
    raise

StatementMeta(, 305dd33b-06f1-4eb7-854a-9828f7493c0a, 16, Finished, Available, Finished, False)

2026-10-04 14:46:25,882 - INFO - Creating gold_dim_table...
2026-10-04 14:46:33,459 - INFO - Query executed


__Validate dim_team__

In [9]:
pri_key_validation("nhl_gold_lakehouse.dbo.gold_dim_team", "team_id")

StatementMeta(, 305dd33b-06f1-4eb7-854a-9828f7493c0a, 17, Finished, Available, Finished, False)

2026-10-04 14:46:39,580 - INFO - Prepare to validate nhl_gold_lakehouse.dbo.gold_dim_team...
2026-10-04 14:46:39,694 - INFO - 1/3 VALIDATION SUCCESS: nhl_gold_lakehouse.dbo.gold_dim_team is created successfully.
2026-10-04 14:46:39,695 - INFO - 1/3 - Existence Check completed
2026-10-04 14:46:41,768 - INFO - 2/3 VALIDATION SUCCESS: team_id is unique
2026-10-04 14:46:41,769 - INFO - 2/3 - Unique check completed.
2026-10-04 14:46:42,642 - INFO - 3/3 VALIDATION SUCCESS: No nulls in team_id
2026-10-04 14:46:42,642 - INFO - 3/3 - Null check completed.
2026-10-04 14:46:42,643 - INFO - Full validation completed.


---
__Create dim_venue__
- Table is meant to be rerun or replace again when fed with new data.
- Venues are associated with multiple timezone id. Rather than using venue as primary key, venue and timezone id are used as composite key. Surrogate keys are assigned to each pair. Refer to the inline comments for more info.

In [11]:
logger.info("Creating dim_venue...")

dim_venue_table_query="""
CREATE OR REPLACE TABLE nhl_gold_lakehouse.dbo.gold_dim_venue AS

WITH v AS(
/*Issue 1: Some venues have multiple timezones - inherited from source. 
Preserving the venue and venue_time_zone_id as a pair for composite key, allowing updates to be made next time should either venue or timezone needs to be updated next time*/
    
    SELECT
        venue,
        venue_time_zone_id,
        count(venue_time_zone_id) OVER (PARTITION BY venue_time_zone_id) AS venue_id_count
    FROM nhl_silver_lakehouse.dbo.silver_game
    GROUP BY venue, venue_time_zone_id
--    ORDER BY venue
)
,
v_default_timezone AS(

SELECT
    venue,
    venue_time_zone_id,
    venue_id_count,
    ROW_NUMBER() OVER(PARTITION BY venue ORDER BY venue_id_count DESC) AS venue_freq_rank
FROM v
--GROUP BY venue, venue_time_zone_id, venue_id_count
)
,
v_key AS(
/* numbering the distinct venue and timezone id pair to form surrogate keys */
SELECT
    row_number() OVER (ORDER BY venue) AS venue_key,
    venue,
    venue_time_zone_id
FROM v_default_timezone
)
,
v_key_cross AS(
/*
Issue 2: As venue have both standard time and daylight time values(one hour behind), there are 2 rows for each venue. 
Below extracting respective values to standard time values and daylight time values each in a column
*/

SELECT 
    vk.venue_key,
    g.venue,
    vk.venue_time_zone_id,
    MAX(CASE WHEN g.venue_time_zone_tz LIKE '_ST' THEN g.venue_time_zone_tz END) AS st_tz, --standard time tab
    MAX(CASE --redefining the timezone as per fact
        WHEN g.venue_time_zone_tz = 'EST' THEN -5
        WHEN g.venue_time_zone_tz = 'CST' THEN -6
        WHEN g.venue_time_zone_tz = 'MST' THEN -7
        WHEN g.venue_time_zone_tz = 'PST' THEN -8
        ELSE NULL END) AS st_tz_offset_redef,
    MAX(CASE WHEN g.venue_time_zone_tz LIKE '_DT' THEN g.venue_time_zone_tz END) AS dt_tz,--daylight time tab
    MAX(CASE --redefining the timezone as per fact
        WHEN g.venue_time_zone_tz = 'EDT' THEN -4
        WHEN g.venue_time_zone_tz = 'CDT' THEN -5
        WHEN g.venue_time_zone_tz = 'MDT' THEN -6
        WHEN g.venue_time_zone_tz = 'PDT' THEN -7
        ELSE NULL END) AS dt_tz_offset_redef
FROM nhl_silver_lakehouse.dbo.silver_game AS g
JOIN v_key AS vk 
ON g.venue = vk.venue AND g.venue_time_zone_id = vk.venue_time_zone_id
GROUP BY vk.venue_key, g.venue, vk.venue_time_zone_id -- Using groupby and max to remove duplicates
)
,
v_key_cross_fillna_xphoenix AS(
SELECT
/* Issue 3: Output table above resulted in some nulls due to venue not used during the daylight/standard timezone. 
This queries fill the nulls with relevant information from other columns for a complete table without nulls.
Phoenix is explicitly excluded because it does not have any timezone changes.It will be handled in the next query 
*/
    venue_key,
    venue,
    venue_time_zone_id,
    CASE --filling standard time nulls with derived value from daylight time columns
        WHEN st_tz IS NULL AND venue_time_zone_id != 'America/Phoenix' THEN LEFT(dt_tz,1) || 'ST' ELSE st_tz END AS stdtime_timezone, 
    CASE -- filling standard time offset nulls based on fact that daylight is 1 hour behind standard time
        WHEN st_tz_offset_redef IS NULL AND venue_time_zone_id != 'America/Phoenix' THEN dt_tz_offset_redef -1 ELSE st_tz_offset_redef END AS stdtime_offset, 
    CASE --same for daylight time nulls
         WHEN dt_tz IS NULL AND venue_time_zone_id != 'America/Phoenix' THEN LEFT(st_tz,1) || 'DT' ELSE dt_tz END AS daylight_timezone,
    CASE -- same for daylight time nulls
        WHEN dt_tz_offset_redef IS NULL AND venue_time_zone_id != 'America/Phoenix' THEN st_tz_offset_redef +1 ELSE dt_tz_offset_redef END AS daylight_offset
FROM v_key_cross
)

SELECT 
/* Issue 4: Phoenix is hardcoded as it changes from MST to PDT, different from the formula used to derive the other column*/

    venue_key,
    venue,
    venue_time_zone_id,
    CASE WHEN stdtime_timezone IS NULL AND venue_time_zone_id = 'America/Phoenix' THEN 'MST' ELSE stdtime_timezone END AS stdtime_timezone,
    CASE WHEN stdtime_offset IS NULL AND venue_time_zone_id = 'America/Phoenix' THEN '-7' ELSE stdtime_offset END AS stdtime_offset,
    CASE WHEN daylight_timezone IS NULL AND venue_time_zone_id = 'America/Phoenix' THEN 'PDT' ELSE daylight_timezone END AS daylight_timezone,
    CASE WHEN daylight_offset IS NULL AND venue_time_zone_id = 'America/Phoenix' THEN '-7' ELSE daylight_offset END AS daylight_offset
FROM v_key_cross_fillna_xphoenix

"""

try: 
    spark.sql(dim_venue_table_query)
    logger.info("Query executed.")
except AnalysisException as e:
    logger.error(f"ANALYSIS EXCEPTION ERROR: Check source table or column name. \n\n Details:{e}")
    raise 
except Exception as e:
    logger.error(f"UNEXPECTED ERROR:\n\n {e}.", exc_info=True)
    raise

StatementMeta(, 305dd33b-06f1-4eb7-854a-9828f7493c0a, 19, Finished, Available, Finished, False)

2026-10-04 14:54:44,148 - INFO - Creating dim_venue...
2026-10-04 14:54:52,331 - INFO - Query executed.


__Validate dim_venue__

In [12]:
pri_key_validation("nhl_gold_lakehouse.dbo.gold_dim_venue", "venue_key")

StatementMeta(, 305dd33b-06f1-4eb7-854a-9828f7493c0a, 20, Finished, Available, Finished, False)

2026-10-04 14:55:04,320 - INFO - Prepare to validate nhl_gold_lakehouse.dbo.gold_dim_venue...
2026-10-04 14:55:04,424 - INFO - 1/3 VALIDATION SUCCESS: nhl_gold_lakehouse.dbo.gold_dim_venue is created successfully.
2026-10-04 14:55:04,425 - INFO - 1/3 - Existence Check completed
2026-10-04 14:55:07,422 - INFO - 2/3 VALIDATION SUCCESS: venue_key is unique
2026-10-04 14:55:07,423 - INFO - 2/3 - Unique check completed.
2026-10-04 14:55:08,481 - INFO - 3/3 VALIDATION SUCCESS: No nulls in venue_key
2026-10-04 14:55:08,481 - INFO - 3/3 - Null check completed.
2026-10-04 14:55:08,482 - INFO - Full validation completed.


---

__Create dim_date__

Included as many date format as possible for reusing the date time for other purpose. The date range is wide in consideration of the longevity of the use of datetable.

In [13]:
logger.info("Creating dim_date table...")

dim_date_query = """
CREATE OR REPLACE TABLE nhl_gold_lakehouse.dbo.gold_dim_date AS

SELECT
    CAST(DATE_FORMAT(date_range, 'yyyyMMdd') AS INT) AS date_key,
    date_range as date,--- already define in the subquery
    CAST(DAY(date_range) AS INT) AS day,
    CAST(DATE_FORMAT(date_range, 'EEE') AS STRING) AS day_of_week,
    CAST(DAYOFWEEK(date_range) AS INT) AS day_number,
    CAST(WEEKOFYEAR(date_range) AS INT) AS week_number,
    CAST(DATE_FORMAT(date_range, 'MMM') AS STRING) AS month,
    CAST(MONTH(date_range) AS INT) AS month_number,
    CAST(QUARTER(date_range) AS INt) AS quarter,
    CAST(YEAR(date_range) AS INT) AS year
FROM 
(SELECT EXPLODE(
        SEQUENCE(
            CAST('1990-01-01' AS DATE), 
            CAST('2050-12-31' AS DATE), 
            INTERVAL 1 DAY
            )
            ) AS date_range )
"""
try:
    spark.sql(dim_date_query)
    logger.info("Query executed.")
    
    table_exists = spark.catalog.tableExists("nhl_gold_lakehouse.dbo.gold_dim_date")
    if table_exists:
        logger.info(f"VALIDATION SUCCESS: nhl_gold_lakehouse.dbo.gold_dim_date is created successfully.")
    else:
        logger.warning(f"1/3 VALIDATION FAIL: nhl_gold_lakehouse.dbo.gold_dim_date is not found. Unable to proceed", exc_info=True)
        raise RuntimeError (f"nhl_gold_lakehouse.dbo.gold_dim_date is not found.")
    logger.info(f"Existence Check completed")

except AnalysisException as e:
    logger.warning(f"ANALYSIS EXCEPTION ERROR: \n\n Details: {e}")
    raise
except Exception as e:
    logger.warning(f"UNEXPECTED ERROR: \n\n {e}.", exc_info=True)
    raise

StatementMeta(, 305dd33b-06f1-4eb7-854a-9828f7493c0a, 21, Finished, Available, Finished, False)

2026-10-04 14:55:56,459 - INFO - Creating dim_date table...
2026-10-04 14:56:02,328 - INFO - Query executed.
2026-10-04 14:56:02,404 - INFO - VALIDATION SUCCESS: nhl_gold_lakehouse.dbo.gold_dim_date is created successfully.
2026-10-04 14:56:02,405 - INFO - Existence Check completed


__Validate dim_date__  
As dim_date table is self created, it requires a different check as follows:

In [14]:
logger.info("Checking for game date(at source) out of gold_dim_date date range...")

date_range_check_query="""
SELECT date_time_GMT
FROM nhl_silver_lakehouse.dbo.silver_game
WHERE date_time_GMT NOT BETWEEN 
    (SELECT MIN(date)FROM nhl_gold_lakehouse.dbo.gold_dim_date) AND
    (SELECT MAX(date)FROM nhl_gold_lakehouse.dbo.gold_dim_date)
"""

try:
    date_range_check_query_result = spark.sql(date_range_check_query)
    if date_range_check_query_result.count() == 0:
        logger.info("VALIDATION SUCCESS: All game date range is within gold_dim_date date range")
    else:
        logger.warning("VALIDATION FAIL: Dates not within range. Details:")
        date_range_check_query_result.show()
        raise ValueError("Dates not within gold_dim_date date range")
except AnalysisException as e:
    logger.error(f"ANALYSIS EXCEPTION ERROR: \n\n Details: {e}")
    raise
except Exception as e:
    logger.error(f"UNEXPECTED ERROR: \n\n {e}", exc_info=True)
    raise
logger.info("Date boundary validation complete")

StatementMeta(, 305dd33b-06f1-4eb7-854a-9828f7493c0a, 23, Finished, Available, Finished, False)

2026-10-04 14:56:48,542 - INFO - Checking for game date(at source) out of gold_dim_date date range...
2026-10-04 14:56:54,266 - INFO - VALIDATION SUCCESS: All game date range is within gold_dim_date date range
2026-10-04 14:56:54,266 - INFO - Date boundary validation complete


--- 
__Create fact_game__  
Summary of changes from silver table to gold table:  
- Added match info from silver_game, team performance from silver_games_teans_stats
- Amended datetime format from upstream to a readable format, 
- Converted 'home_or_away' column, 'settled_in' to flag-type, 
- Dropped 'outcome' as it is a repeated information.
- Added in opposing team's performance
- Derived goal differences
- Joined with dim_venue

In [16]:
logger.info("Creating gold_fact_game table...")

gold_fact_game_query = """

CREATE OR REPLACE TABLE nhl_gold_lakehouse.dbo.gold_fact_game AS

/*Creation of base table by combining the relevant columns all in the team level grain. 
Opponent, opponent goals, tie flag, difference in goal can be derived from team_id in self-join in the next stage */

WITH A AS(SELECT
    g.season,
    g.date_time_GMT,
    g.game_id,
    gts.team_id,
    CAST(CASE -- convert to flag
        WHEN gts.home_or_away = 'home' THEN 1
        ELSE 0
    END AS INT) AS home_flag,
    CAST(CASE -- convert to flag
        WHEN gts.home_or_away = 'away' THEN 1
        ELSE 0
    END AS INT)AS away_flag,
    v.venue_key,
    g.type,
    gts.settled_in,
    gts.head_coach,
    gts.start_rink_side,
    gts.goals,
    gts.shots,
    gts.hits,
    gts.penalty_in_minutes,
    gts.power_play_opportunities,
    gts.power_play_goals,
    gts.face_off_win_percentage,
    gts.giveaways,
    gts.takeaways,
    gts.blocked,
    gts.won AS win_flag,
    CAST(CASE  -- convert to flag
        WHEN settled_in = 'REG' THEN 1
        ELSE 0
    END AS INT) AS regular_flag,
    CAST(CASE  -- convert to flag
        WHEN settled_in = 'OT' THEN 1
        ELSE 0 
    END AS INT)AS overtime_flag
FROM nhl_silver_lakehouse.dbo.silver_game as g
JOIN nhl_silver_lakehouse.dbo.silver_game_teams_stats as gts
ON g.game_id = gts.game_id
JOIN nhl_gold_lakehouse.dbo.gold_dim_venue AS v
ON (v.venue = g.venue) AND (v.venue_time_zone_id = g.venue_time_zone_id)
)
, B AS(

/*Adding in opponent, opponent goals by using home / away team flag with a self-join. 
This is done by identifying the home team first and self-join the same table with only away team on the same game_id as opponent team, 
As only team playing as home_team are idenfitied, the reversed(identify away team and find opponent team id/goal) is also identified,
so that the row with the same team playing as away-team will not be missed out. The latter is appended with the use of union. 

In total of 47,462 rows - same number of rows as the table A in stage 1*/    

SELECT 
    a.season,
    a.date_time_GMT,
    a.game_id,
    a.team_id,
    a.home_flag,
    a.away_flag,
    a1.team_id AS opponent_team, -- new column added in self-join
    a.venue_key,
    a.type,
    a.settled_in,
    a.head_coach,
    a.start_rink_side,
    a.goals,
    a.shots,
    a.hits,
    a.penalty_in_minutes,
    a.power_play_opportunities,
    a.power_play_goals,
    a.face_off_win_percentage,
    a.giveaways,
    a.takeaways,
    a.blocked,
    a1.goals AS opponent_goals, -- new column added in self-join
    a1.hits AS opponent_hits,
    a1.shots AS opponent_shots,
    a1.blocked AS opponent_blocked,
    a.win_flag,
    a.regular_flag,
    a.overtime_flag
FROM A as a
JOIN (SELECT
        game_id,
        team_id,
        home_flag,
        away_flag,
        goals,
        hits,
        shots,
        blocked
        FROM A
        WHERE away_flag = 1) AS a1
ON a.game_id = a1.game_id
WHERE a.home_flag = 1

UNION

SELECT 
    a.season,
    a.date_time_GMT,
    a.game_id,
    a.team_id,
    a.home_flag,
    a.away_flag,
    a1.team_id AS opponent_team,
    a.venue_key,
    a.type,
    a.settled_in,
    a.head_coach,
    a.start_rink_side,
    a.goals,
    a.shots,
    a.hits,
    a.penalty_in_minutes,
    a.power_play_opportunities,
    a.power_play_goals,
    a.face_off_win_percentage,
    a.giveaways,
    a.takeaways,
    a.blocked,
    a1.goals AS opponent_goals,
    a1.hits AS opponent_hits,
    a1.shots AS opponent_shots,
    a1.blocked AS opponent_blocked,
    a.win_flag,
    a.regular_flag,
    a.overtime_flag
FROM A as a
JOIN (SELECT
        game_id,
        team_id,
        home_flag,
        away_flag,
        goals,
        hits,
        shots,
        blocked
        FROM A
        WHERE home_flag = 1) AS a1
ON a.game_id = a1.game_id
WHERE a.away_flag = 1
)

SELECT

/*Deriving 'goal difference' and 'tie_flag' from the newly added column - opponent goals*/    

    type,
    season AS season_key,
    DATE_FORMAT(date_time_GMT, 'yyyy-MM-dd HH:mm:ss') as date_timeGMT, -- convert date format for easy reading
    CAST(DATE_FORMAT(date_time_GMT, 'yyyy-MM-dd') AS DATE) as date_GMT, -- convert date format to match
    CAST(DATE_FORMAT(date_time_GMT, 'yyyyMMdd') AS INT) as date_key, -- convert to datekey format to join
    venue_key,
    start_rink_side,
    game_id,
    team_id,
    home_flag, 
    away_flag,
    opponent_team AS opponent_team_id,
    settled_in,
    head_coach,
    goals,
    shots,
    hits,
    blocked,
    penalty_in_minutes,
    power_play_opportunities,
    power_play_goals,
    CAST(face_off_win_percentage AS DECIMAL(5,2)) AS face_off_win_percent,
    giveaways,
    takeaways,
    opponent_goals,
    CAST(goals - opponent_goals AS INT) AS goals_more_than_opponent,
    CAST(CASE
        WHEN win_flag = True THEN 1
        ELSE 0
    END AS INT) as win_flag,
    CAST(CASE
        WHEN goals = opponent_goals THEN 1
        ELSE 0
    END AS INT) AS tie_flag,
    opponent_hits,
    opponent_shots,
    opponent_blocked,
    regular_flag,
    overtime_flag 
FROM B
ORDER BY season, game_id

"""

try:
    spark.sql(gold_fact_game_query)
    logger.info("Query executed")

except AnalysisException as e:
    logger.error(f"ANALYSIS EXCEPTION ERROR: Check source table or column name. \n\n Details:{e} ") 
    raise 
except Exception as e:
    logger.error(f"UNEXPECTED ERROR:\n\n {e}.", exc_info=True) 
    raise

StatementMeta(, 305dd33b-06f1-4eb7-854a-9828f7493c0a, 26, Finished, Available, Finished, False)

2026-10-04 15:01:37,803 - INFO - Creating gold_fact_game table...
2026-10-04 15:01:48,737 - INFO - Query executed


__Validate fact_game__  
gold_fact_game are created based on silver_games_teams_stats and silver_game, it is validated by comparing with silver_game_teams_stats mainly as majority of the stats are from this table. 

In [17]:
logger.info("Begin validation...")

distinct_game_id_count_two_query= """
    SELECT
        game_id, 
        COUNT(*) 
    FROM nhl_gold_lakehouse.dbo.gold_fact_game 
    GROUP BY game_id 
    HAVING COUNT(*) !=2
    """

try:
    #table existence validation
    table_exists = spark.catalog.tableExists("nhl_gold_lakehouse.dbo.gold_fact_game")
    if table_exists:
        logger.info("VALIDATION SUCCESS: gold_fact_game table is successfully created")
    else:
        logger.warning("VALIDATION FAIL: No such table found.")
        raise RuntimeError("gold_fact_game is not found in catalog")
    
    #row validation - row count before vs after
    gold_fact_game_row_count = spark.table("nhl_gold_lakehouse.dbo.gold_fact_game").count()
    silver_game_teams_stats_row_count = spark.table("nhl_silver_lakehouse.dbo.silver_game_teams_stats").count()
    if gold_fact_game_row_count == silver_game_teams_stats_row_count:
        logger.info("VALIDATION SUCCESS: Row counts before = row count after")
    else:
       logger.warning(f"VALIDATION FAIL: Row counts after - gold_fact_game ({gold_fact_game_row_count}) is not = row count before - silver_game_teams_stats ({silver_game_teams_stats_row_count})")
       raise ValueError ("Row counts does not match between gold_fact_game and silver_game_teams_stats")
    
    #number of game id should be half of the row count (2 team id for 1 game id)
    gold_fact_game_distinct_game_id = spark.table("nhl_gold_lakehouse.dbo.gold_fact_game").select("game_id").distinct().count()
    if gold_fact_game_row_count == 2*gold_fact_game_distinct_game_id:
        logger.info("VALIDATION SUCCESS: Met the expectation of total row counts = 2 x number of distinct game id")
    else:
       logger.warning(f"VALIDATION FAIL: Total row counts ({gold_fact_game_row_count}) does not tally with distinct game id ({gold_fact_game_distinct_game_id})")
       raise ValueError ("Did not meet the expectation of row counts = 2 x number of distinct game id")

    #2 count of game id
    distinct_game_id_not_two_result = spark.sql(distinct_game_id_count_two_query)
    if distinct_game_id_not_two_result.count() ==0:
        logger.info("VALIDATION SUCCESS: All game_ids met the expectation of having total count = 2")
    else:
        logger.warning(f"VALIDATION FAIL: There are {distinct_game_id_not_two_result.count()} that does not have total count of 2")
        raise ValueError("Not all game_ids have total count of 2")

except AnalysisException as e:
    logger.error(f"ANALYSIS EXCEPTION ERROR: Check source table or column name. \n\n Details:{e} ") 
    raise 
except Exception as e:
    logger.error(f"UNEXPECTED ERROR:\n\n {e}.", exc_info=True) 
    raise
        

StatementMeta(, 305dd33b-06f1-4eb7-854a-9828f7493c0a, 27, Finished, Available, Finished, False)

2026-10-04 15:02:30,322 - INFO - Begin validation...
2026-10-04 15:02:30,449 - INFO - VALIDATION SUCCESS: gold_fact_game table is successfully created
2026-10-04 15:02:33,327 - INFO - VALIDATION SUCCESS: Row counts before = row count after
2026-10-04 15:02:35,088 - INFO - VALIDATION SUCCESS: Met the expectation of total row counts = 2 x number of distinct game id
2026-10-04 15:02:35,939 - INFO - VALIDATION SUCCESS: All game_ids met the expectation of having total count = 2
